In [0]:
%run ../connection

In [0]:
%run ./silver_utils

In [0]:
%run ./config

In [0]:

 
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, LongType, StringType, TimestampType
 
adls_name = "adlsnewhp1"
init_silver_config(adls_name)
 
logger = get_logger("silver_hourly_timeseries")
 
logger.info("=" * 70)
logger.info("Silver: hourly_timeseries — START")
logger.info(f"  Run ID    : {SilverConfig.RUN_ID}")
logger.info(f"  Bronze in : {BronzePaths.MARKET_CHART}")
logger.info(f"  Silver out: {SilverPaths.HOURLY_TIMESERIES}")
logger.info("=" * 70)
 

In [0]:

 
bronze_df, max_bronze_ts = read_bronze_incremental(
    spark, BronzePaths.MARKET_CHART, WatermarkPaths.MARKET_CHART,
    WatermarkPaths.WATERMARK_TABLE, logger
)
 
assert_required_columns(
    bronze_df,
    required_cols=["coin_id", "payload", "pipeline_run_id"],
    table_name="market_chart_raw",
    logger=logger
)

In [0]:
bronze_df.display()

In [0]:

# WHY posexplode INSTEAD OF explode?
#   explode() gives the value but not the array position. Without position,
#   we have no reliable key to join the three arrays together (they could
#   theoretically have different lengths or ordering if the API had a bug).
#   posexplode() gives position explicitly, making the join deterministic.

 
logger.info("CELL 3: Exploding three parallel arrays with position index")
 

prices_df = (
    bronze_df
    .select(
        F.col("coin_id").cast(StringType()),
        F.posexplode(F.col("payload.prices")).alias("pos", "pair")
    )
    .select(
        "coin_id",
        "pos",

        F.from_unixtime((F.col("pair")[0] / 1000).cast("long"))
         .cast(TimestampType())
         .alias("hour_timestamp"),
        F.col("pair")[1].cast(DoubleType()).alias("price_usd"),
    )
)
 

market_caps_df = (
    bronze_df
    .select(
        F.col("coin_id").cast(StringType()),
        F.posexplode(F.col("payload.market_caps")).alias("pos", "pair")
    )
    .select(
        "coin_id",
        "pos",
        F.col("pair")[1].cast(LongType()).alias("market_cap_usd"),
    )
)
 

volumes_df = (
    bronze_df
    .select(
        F.col("coin_id").cast(StringType()),
        F.posexplode(F.col("payload.total_volumes")).alias("pos", "pair")
    )
    .select(
        "coin_id",
        "pos",
        F.col("pair")[1].cast(LongType()).alias("volume_usd"),
    )
)
 
logger.info(f"  prices rows: {prices_df.count():,}")
logger.info(f"  market_cap rows: {market_caps_df.count():,}")
logger.info(f"  volume rows: {volumes_df.count():,}")
 
 

In [0]:

 
logger.info("CELL 4: Joining three arrays on coin_id + pos")
 
joined_df = (
    prices_df
    .join(
        market_caps_df,
        on  = ["coin_id", "pos"],
        how = "inner"
    )
    .join(
        volumes_df,
        on  = ["coin_id", "pos"],
        how = "inner"
    )
    .drop("pos")    # pos was only needed for the join, not for the Silver table
    .withColumn("silver_processed_timestamp", get_silver_timestamp(SilverConfig.RUN_TS))
)
 
raw_count = joined_df.count()
logger.info(f"  Rows after join: {raw_count:,}")
 

In [0]:
joined_df.display()

In [0]:

# DATA QUALITY FILTERS

 
logger.info("CELL 5: Applying data quality filters")
 
clean_df = (
    joined_df
    .filter(F.col("coin_id").isNotNull())
    .filter(F.col("hour_timestamp").isNotNull())
    # Price must exist and be positive
    .filter(
        F.col("price_usd").isNotNull() &
        (F.col("price_usd") > 0.0)
    )

    .filter(
        F.col("volume_usd").isNull() |
        (F.col("volume_usd") >= DataQuality.MIN_VOLUME_USD)
    )
)
 
clean_count = clean_df.count()
logger.info(f"  Rows after quality filters: {clean_count:,}")
 
validate_drop_rate(
    rows_before  = raw_count,
    rows_after   = clean_count,
    max_fraction = DataQuality.MAX_DROP_FRACTION,
    table_name   = "hourly_timeseries",
    logger       = logger,
)
 

In [0]:

# DEDUPLICATE
logger.info("CELL 6: Deduplicating on coin_id + hour_timestamp")
 
dedup_df    = clean_df.dropDuplicates(MergeKeys.HOURLY_TIMESERIES)
dedup_count = dedup_df.count()
logger.info(f"  Rows after dedup: {dedup_count:,} (dropped {clean_count - dedup_count:,})")

In [0]:
 

# FINAL COLUMN REORDER

 
final_df = dedup_df.select(*SilverColumns.HOURLY_TIMESERIES)
log_schema(final_df, "hourly_timeseries_final", logger)
 

In [0]:
 

 
logger.info("CELL 8: MERGE into silver/hourly_timeseries")
 
merge_stats = delta_merge(
    spark      = spark,
    new_df     = final_df,
    table_path = SilverPaths.HOURLY_TIMESERIES,
    merge_keys = MergeKeys.HOURLY_TIMESERIES,
    logger     = logger,
)

update_watermark(
    spark          = spark,
    source_table   = WatermarkPaths.MARKET_CHART,
    new_ts         = max_bronze_ts,
    watermark_path = WatermarkPaths.WATERMARK_TABLE,
    run_ts         = SilverConfig.RUN_TS,
    logger         = logger,
)
 

In [0]:

 
logger.info("CELL 9: OPTIMIZE silver/hourly_timeseries")
 
spark.sql(f"OPTIMIZE delta.`{SilverPaths.HOURLY_TIMESERIES}` ZORDER BY (coin_id, hour_timestamp)")
logger.info("  ✓ OPTIMIZE complete")
 

In [0]:

 
summary = {
    "notebook"                 : "02c_silver_hourly_timeseries",
    "pipeline_run_id"          : SilverConfig.RUN_ID,
    "run_timestamp_utc"        : SilverConfig.RUN_TS.isoformat(),
    "bronze_source"            : BronzePaths.MARKET_CHART,
    "silver_target"            : SilverPaths.HOURLY_TIMESERIES,
    "rows_from_bronze"         : raw_count,
    "rows_after_quality_filter": clean_count,
    "rows_after_dedup"         : dedup_count,
    "merge_rows_before"        : merge_stats["rows_before"],
    "merge_rows_after"         : merge_stats["rows_after"],
    "merge_rows_inserted"      : merge_stats["rows_inserted"],
    "status"                   : "SUCCESS",
}
 
write_run_log(summary, LogPaths.HOURLY_TIMESERIES, logger)
 
logger.info("=" * 70)
logger.info("Silver: hourly_timeseries — COMPLETE")
for k, v in summary.items():
    logger.info(f"  {k:<35}: {v}")
logger.info("=" * 70)